In [ ]:
# Install Mazinger and start a local, OpenAI-compatible Homura server.
import os
import sys
import time
import json
import shutil
import zipfile
import subprocess
import urllib.request
import signal
from pathlib import Path

os.makedirs("/kaggle/working", exist_ok=True)
os.environ["OPENAI_API_KEY"] = "EMPTY"
os.environ["OPENAI_BASE_URL"] = "http://localhost:8000/v1"
os.environ["OPENAI_MODEL"] = "IndexTeam/Index-Homura-2B"
os.environ["MAZINGER_DISABLE_VISION"] = "1"
os.environ["MAZINGER_LLM_MAX_OUTPUT_TOKENS"] = "2048"

archive_candidates = list(Path("/kaggle/input").rglob("mazinger_source.zip"))
mazinger_extract_path = Path("/kaggle/working/mazinger")
search_root = Path("/kaggle/input")
if archive_candidates:
    if mazinger_extract_path.exists():
        shutil.rmtree(mazinger_extract_path)
    mazinger_extract_path.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_candidates[0]) as source_archive:
        extraction_root = mazinger_extract_path.resolve()
        for member in source_archive.infolist():
            destination = (extraction_root / member.filename).resolve()
            if destination != extraction_root and extraction_root not in destination.parents:
                raise RuntimeError("Mazinger source archive contains an unsafe path.")
        source_archive.extractall(extraction_root)
    search_root = mazinger_extract_path
else:
    print("Kaggle mounted the source archive as an extracted folder; locating pyproject.toml directly.", flush=True)
mazinger_candidates = [
    path.parent for path in search_root.rglob("pyproject.toml")
    if (path.parent / "mazinger").is_dir()
    or (path.parent / "src" / "mazinger").is_dir()
    or path.parent.name.lower() in {"mazinger", "mazinger_source"}
]
if not mazinger_candidates:
    mounted_files = [str(p) for p in Path("/kaggle/input").rglob("pyproject.toml")]
    raise FileNotFoundError(f"Mazinger pyproject.toml not found in Kaggle input. Found: {mounted_files}")
mazinger_path = str(mazinger_candidates[0])
# Install the runtime libraries and import the bundled source directly (no wheel build needed).
os.environ["PYTHONPATH"] = mazinger_path + os.pathsep + os.environ.get("PYTHONPATH", "")
subprocess.run([
    sys.executable, "-m", "pip", "install", "--no-cache-dir",
    "yt-dlp>=2026.3.17", "openai>=1.0", "json-repair>=0.28", "Pillow>=10.0",
    "soundfile>=0.12", "numpy>=1.24", "tqdm>=4.60", "python-slugify>=8.0",
    "faster-whisper", "av>=14.0.0", "demucs", "omnivoice"
], check=True)
subprocess.run([sys.executable, "-c", "import mazinger; print('Mazinger source import OK')"], check=True)

# vLLM and speech dependency downloads can take several minutes on a fresh Kaggle session.
# On Kaggle T4 x2, reserve one GPU for ASR/TTS and run Homura-2B on the other.
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "vllm==0.29.0"], check=True)
# Kaggle's newest PyAV release dropped/changed this optional kwarg; faster-whisper still passes it.
# Remove only that compatibility kwarg so audio decode works across PyAV releases.
audio_module = subprocess.check_output(
    [sys.executable, "-c", "import faster_whisper.audio; print(faster_whisper.audio.__file__)"],
    text=True
).strip()
audio_source = Path(audio_module).read_text(encoding="utf-8")
legacy_call = 'av.open(input_file, mode="r", metadata_errors="ignore")'
if legacy_call in audio_source:
    Path(audio_module).write_text(audio_source.replace(legacy_call, 'av.open(input_file, mode="r")'), encoding="utf-8")
    print("Applied PyAV audio decode compatibility patch.", flush=True)
else:
    print("faster-whisper audio decoder has no legacy PyAV call; no patch needed.", flush=True)

try:
    import torch
    gpu_count = torch.cuda.device_count()
except Exception:
    gpu_count = 0
if gpu_count < 1:
    raise RuntimeError("No CUDA GPU is visible. Enable a Kaggle GPU accelerator and rerun.")
tensor_parallel = 1
if gpu_count >= 2:
    print("Using Homura-2B on GPU 0 and reserving GPU 1 for ASR/TTS.", flush=True)
else:
    print("Only one GPU is visible; using Homura-2B on the available GPU.", flush=True)
print(f"Visible CUDA GPUs: {gpu_count}; vLLM tensor parallel size: {tensor_parallel}", flush=True)

vllm_log = open("/kaggle/working/vllm_server.log", "w", encoding="utf-8")
def start_vllm(model_id):
    vllm_cli = shutil.which("vllm")
    if not vllm_cli:
        raise RuntimeError("vLLM CLI was not installed successfully.")
    is_small = model_id.endswith("-2B")
    server_gpu_count = 1
    server_env = os.environ.copy()
    if gpu_count >= 2:
        # Reserve one T4 for speech models later in the pipeline.
        server_env["CUDA_VISIBLE_DEVICES"] = "0"
    command = [
        vllm_cli, "serve", model_id,
        "--served-model-name", model_id,
        "--host", "127.0.0.1", "--port", "8000",
        "--tensor-parallel-size", str(server_gpu_count),
        "--gpu-memory-utilization", "0.72",
        "--max-model-len", "4096",
        "--max-num-seqs", "1",
        "--max-num-batched-tokens", "2048",
        "--dtype", "half",
        "--trust-remote-code",
        "--language-model-only",
    ]
    print(f"Starting vLLM model {model_id}: {' '.join(command)}", flush=True)
    return subprocess.Popen(
        command, stdout=vllm_log, stderr=subprocess.STDOUT,
        env=server_env, start_new_session=True
    )

def stop_vllm(process):
    # Kill the whole process group; vLLM may leave CUDA worker children behind.
    try:
        os.killpg(process.pid, signal.SIGTERM)
    except ProcessLookupError:
        return
    try:
        process.wait(timeout=30)
    except subprocess.TimeoutExpired:
        pass
    time.sleep(2)
    try:
        os.killpg(process.pid, signal.SIGKILL)
    except ProcessLookupError:
        pass

def wait_for_vllm(process, timeout_seconds):
    deadline = time.monotonic() + timeout_seconds
    while time.monotonic() < deadline:
        if process.poll() is not None:
            return False
        try:
            with urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=5) as response:
                if response.status == 200:
                    return True
        except Exception:
            time.sleep(5)
    return False

# Allow up to 20 minutes for model weight download and server startup.
vllm_process = start_vllm(os.environ["OPENAI_MODEL"])
if not wait_for_vllm(vllm_process, 1200):
    stop_vllm(vllm_process)
    vllm_log.flush()
    tail = Path("/kaggle/working/vllm_server.log").read_text(encoding="utf-8", errors="replace")[-30000:]
    raise RuntimeError("Homura-2B did not start within 20 minutes. vLLM log tail:\n" + tail)
print(f"Local Homura server ready at {os.environ['OPENAI_BASE_URL']} using {os.environ['OPENAI_MODEL']}.", flush=True)

In [ ]:
# Task 2: Find the uploaded video in Kaggle input directory
import os
import glob
import shutil
import json

input_dir = "/kaggle/input"
video_path = None
job_config = {"target_language": "Hindi"}
for config_path in Path(input_dir).rglob("dub_job.json"):
    try:
        job_config = json.loads(config_path.read_text(encoding="utf-8"))
        break
    except (OSError, json.JSONDecodeError) as exc:
        print(f"Ignoring unreadable job config {config_path}: {exc}", flush=True)
target_language = str(job_config.get("target_language") or "Hindi")
print(f"Requested target language: {target_language}", flush=True)

for root, dirs, files in os.walk(input_dir):
    for file in files:
        if file.endswith(('.mp4', '.mkv', '.avi', '.mov')):
            video_path = os.path.join(root, file)
            break
    if video_path:
        break

if not video_path:
    raise FileNotFoundError("Video not found in Kaggle input directory!")
print(f"Found video at: {video_path}")

In [ ]:
# Execute Mazinger against the local Homura vLLM server.
import traceback
import subprocess
import sys

try:
    print("\U0001f680 Starting Mazinger Dubbing Pipeline...", flush=True)
    base_dir = "/kaggle/working/mazinger_output"
    os.makedirs(base_dir, exist_ok=True)
    if not os.environ.get("OPENAI_MODEL"):
        raise RuntimeError("Local Homura server was not initialized. Rerun the notebook from the first cell.")
    print(f"LLM provider: local {os.environ['OPENAI_MODEL']} via {os.environ['OPENAI_BASE_URL']}.", flush=True)
    voice_sample_path = "/kaggle/working/source_voice_reference.wav"
    subprocess.run([
        "ffmpeg", "-y", "-ss", "6.9", "-i", video_path, "-t", "9.5",
        "-vn", "-ac", "1", "-ar", "24000", "-c:a", "pcm_s16le", voice_sample_path
    ], check=True, capture_output=True)
    cmd = [
        sys.executable, "-m", "mazinger", "dub", video_path,
        "--tts-engine", "omnivoice", "--voice-sample", voice_sample_path,
        "--target-language", target_language,
        "--output-type", "video",
        "--device", "cuda",
        "--transcribe-method", "faster-whisper",
        "--openai-base-url", "http://localhost:8000/v1",
        "--openai-api-key", "EMPTY",
        "--llm-model", os.environ["OPENAI_MODEL"]
    ]
    print(f"Executing: {' '.join(cmd)}", flush=True)
    audio_env = os.environ.copy()
    if os.environ["OPENAI_MODEL"].endswith("-2B") and torch.cuda.device_count() >= 2:
        audio_env["CUDA_VISIBLE_DEVICES"] = "1"
    result = subprocess.run(cmd, cwd=base_dir, capture_output=True, text=True, encoding="utf-8", errors="replace", env=audio_env)
    print(f"STDOUT:\n{result.stdout}", flush=True)
    print(f"STDERR:\n{result.stderr}", flush=True)
    if result.returncode != 0:
        raise RuntimeError(f"Mazinger CLI failed with return code {result.returncode}")
    print("\u2705 Mazinger finished successfully.", flush=True)

    output_candidates = []
    for root, dirs, files in os.walk(base_dir):
        for file in files:
            if file.lower() == "dubbed.mp4":
                output_candidates.append(os.path.join(root, file))
    output_candidates = [path for path in output_candidates if os.path.getsize(path) > 0]
    output_video = output_candidates[-1] if output_candidates else None
    if output_video:
        shutil.copy(output_video, "/kaggle/working/output.mp4")
        print("Copied final video to /kaggle/working/output.mp4", flush=True)
    else:
        message = "Could not find a non-empty dubbed.mp4 from Mazinger."
        with open("/kaggle/working/error_log.txt", "w", encoding="utf-8") as error_file:
            error_file.write(message + "\n")
    # A non-empty mp4 alone is not proof of a complete movie dub. Check media
    # duration, required streams, and translated subtitle coverage before
    # reporting success. Feature films are expected to preserve the full source
    # timeline, including quiet scenes and end credits.
    report_data = {
        "Status": "Failed",
        "Target_Language": target_language,
        "Output_File": os.path.basename(output_video) if output_video else "Missing",
        "WER": "Not measured", "Sync_Offset": "Not measured", "Speaker_Similarity": "Not measured",
        "Validation": {"passed": False, "checks": []},
    }
    if output_video:
        def media_info(path):
            result = subprocess.run([
                "ffprobe", "-v", "error", "-show_entries", "format=duration:stream=codec_type",
                "-of", "json", path
            ], check=True, capture_output=True, text=True)
            return json.loads(result.stdout)

        source_info = media_info(video_path)
        output_info = media_info(output_video)
        source_duration = float(source_info.get("format", {}).get("duration", 0))
        output_duration = float(output_info.get("format", {}).get("duration", 0))
        stream_types = {stream.get("codec_type") for stream in output_info.get("streams", [])}
        tolerance = max(2.0, source_duration * 0.002)
        duration_ok = source_duration > 0 and output_duration > 0 and abs(source_duration - output_duration) <= tolerance
        streams_ok = "video" in stream_types and "audio" in stream_types

        project_root = Path(base_dir)
        translated_candidates = list(project_root.rglob("translated.raw.srt"))
        if not translated_candidates:
            translated_candidates = list(project_root.rglob("translated.srt"))
        translated_srt = translated_candidates[-1] if translated_candidates else None
        translated_text = translated_srt.read_text(encoding="utf-8", errors="replace") if translated_srt else ""
        subtitle_blocks = sum(1 for line in translated_text.splitlines() if "-->" in line)
        source_candidates = list(project_root.rglob("source.raw.srt"))
        source_text = source_candidates[-1].read_text(encoding="utf-8", errors="replace") if source_candidates else ""
        source_blocks = sum(1 for line in source_text.splitlines() if "-->" in line)
        coverage = subtitle_blocks / source_blocks if source_blocks else 0.0
        # Hindi output should contain Devanagari text in the translated subtitle
        # artifact. This catches the prior case where source-language text was
        # accidentally left unchanged while the job was labeled successful.
        devanagari = sum(1 for char in translated_text if "\u0900" <= char <= "\u097f")
        letters = sum(1 for char in translated_text if char.isalpha())
        script_ratio = devanagari / letters if letters else 0.0
        hindi_script_ok = target_language.lower() != "hindi" or script_ratio >= 0.15
        subtitles_ok = bool(translated_srt and source_blocks > 0 and coverage >= 0.80 and hindi_script_ok)
        checks = [
            {"name": "full_timeline_duration", "passed": duration_ok, "source_seconds": round(source_duration, 2), "output_seconds": round(output_duration, 2), "allowed_drift_seconds": round(tolerance, 2)},
            {"name": "video_and_audio_streams", "passed": streams_ok, "streams": sorted(stream_types)},
            {"name": "translated_subtitle_coverage", "passed": subtitles_ok, "source_segments": source_blocks, "translated_segments": subtitle_blocks, "coverage": round(coverage, 4), "devanagari_ratio": round(script_ratio, 4) if target_language.lower() == "hindi" else None},
        ]
        report_data["Validation"] = {"passed": all(check["passed"] for check in checks), "checks": checks}
        report_data["Source_Duration_Seconds"] = round(source_duration, 2)
        report_data["Output_Duration_Seconds"] = round(output_duration, 2)
        if report_data["Validation"]["passed"]:
            report_data["Status"] = "Success"
    with open("/kaggle/working/report.json", "w", encoding="utf-8") as report_file:
        json.dump(report_data, report_file, ensure_ascii=False, indent=2)
    if not output_video:
        raise RuntimeError("Mazinger exited without a non-empty dubbed.mp4 output; inspect error_log.txt.")
    if report_data["Status"] != "Success":
        raise RuntimeError("Output failed full-duration/media/subtitle quality checks. Inspect report.json and artifacts; it will not be reported as a completed dub.")
except Exception:
    error_text = traceback.format_exc()
    print(f"\u274c Error occurred!\n{error_text}", flush=True)
    with open("/kaggle/working/error_log.txt", "w", encoding="utf-8") as error_file:
        error_file.write(error_text)
    with open("/kaggle/working/report.json", "w", encoding="utf-8") as report_file:
        json.dump({"Status": "Failed", "Target_Language": target_language, "Output_File": "Missing", "WER": "Not measured", "Sync_Offset": "Not measured", "Speaker_Similarity": "Not measured"}, report_file)
    raise
finally:
    if "vllm_process" in globals() and vllm_process.poll() is None:
        vllm_process.terminate()
        try:
            vllm_process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            vllm_process.kill()
    if "vllm_log" in globals():
        vllm_log.flush()
        vllm_log.close()
